# 第 10 章实验：合成迁移学习与 Grad-CAM
先在条纹图上学习通用边缘特征，再冻结特征层训练新分类头。全程随机初始化且不下载权重。

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(10)
pretrain_steps = 80
transfer_steps = 50
image_size = 24

In [ ]:
def make_stripes(n, inverted=False):
    images = torch.zeros(n, 1, image_size, image_size)
    labels = torch.arange(n) % 2
    for i, label in enumerate(labels):
        if int(label) == 0:
            images[i, 0, :, 7:11] = 1.0
        else:
            images[i, 0, 7:11, :] = 1.0
    images += 0.12 * torch.randn_like(images)
    images = images.clamp(0, 1)
    return (1 - images if inverted else images), labels

class TinyTransferNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 6, 3, padding=1), nn.ReLU(), nn.Conv2d(6, 8, 3, padding=1), nn.ReLU())
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(8, 2)
    def forward(self, x):
        feature_map = self.features(x)
        return self.head(self.pool(feature_map).flatten(1))

model = TinyTransferNet()
source_x, source_y = make_stripes(64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
for _ in range(pretrain_steps):
    optimizer.zero_grad(); loss = F.cross_entropy(model(source_x), source_y); loss.backward(); optimizer.step()
print('source accuracy:', float((model(source_x).argmax(1) == source_y).float().mean()))

In [ ]:
for parameter in model.features.parameters():
    parameter.requires_grad = False
model.head = nn.Linear(8, 2)
target_x, target_y = make_stripes(12, inverted=True)
optimizer = torch.optim.Adam(model.head.parameters(), lr=0.05)
for _ in range(transfer_steps):
    optimizer.zero_grad(); loss = F.cross_entropy(model(target_x), target_y); loss.backward(); optimizer.step()
trainable = [name for name, p in model.named_parameters() if p.requires_grad]
accuracy = float((model(target_x).argmax(1) == target_y).float().mean())
print('trainable parameters:', trainable)
print('target small-set accuracy:', accuracy)

## Grad-CAM
对最后卷积层的激活保留梯度。热图是相关性线索，不是像素级真值或因果证明。

In [ ]:
saved = {}
def capture_activation(module, inputs, output):
    saved['activation'] = output
    output.retain_grad()
handle = model.features[2].register_forward_hook(capture_activation)
sample = target_x[:1].clone().requires_grad_(True)
model.zero_grad()
logits = model(sample)
class_id = int(logits.argmax(1))
logits[0, class_id].backward()
activation = saved['activation']
weights = activation.grad.mean(dim=(2, 3), keepdim=True)
cam = (weights * activation).sum(dim=1, keepdim=True).relu()
cam = cam - cam.amin(dim=(2, 3), keepdim=True)
cam = cam / cam.amax(dim=(2, 3), keepdim=True).clamp_min(1e-8)
cam = F.interpolate(cam, size=sample.shape[-2:], mode='bilinear', align_corners=False)
handle.remove()
print('logits:', tuple(logits.shape), 'activation:', tuple(activation.shape), 'cam:', tuple(cam.shape))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(sample.detach()[0, 0], cmap='gray'); axes[0].set_title('target sample')
axes[1].imshow(sample.detach()[0, 0], cmap='gray'); axes[1].imshow(cam.detach()[0, 0], cmap='jet', alpha=0.55); axes[1].set_title(f'Grad-CAM class {class_id}')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

### 破坏实验建议
把训练图的背景与类别绑定，再用相反背景测试；比较准确率和热图是否转向背景。该干预比单看一张热图更能发现 shortcut learning。